In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# Olist 데이터 전처리
- 목표: 각 테이블을 개별 전처리한 뒤, 안전하게 조인 가능한 상태로 준비
- 원칙:
  1. 전처리 먼저, 조인은 나중
  2. `orders`를 기준 축(order grain)으로 관리
  3. 다건 테이블(`payments`, `reviews`, `geolocation`)은 집계 후 조인

In [3]:
customers = pd.read_csv("../../../data/olist_customers_dataset.csv")
geolocation = pd.read_csv("../../../data/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../../../data/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../../../data/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../../../data/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../../../data/olist_orders_dataset.csv")
products = pd.read_csv("../../../data/olist_products_dataset.csv")
sellers = pd.read_csv("../../../data/olist_sellers_dataset.csv")
cat_tr = pd.read_csv("../../../data/product_category_name_translation.csv")

## 1) 공통 점검 함수
- 테이블별 행 수, 고유키, 결측치 비율을 빠르게 확인합니다.

In [4]:
def audit_table(df, name, key_cols=None, topn_null=10):
    print(f"\n===== {name} =====")
    print(f"shape: {df.shape}")
    if key_cols:
        for k in key_cols:
            if k in df.columns:
                print(f"unique({k}): {df[k].nunique(dropna=True)} / null: {df[k].isna().sum()}")
    null_rate = (df.isna().mean() * 100).sort_values(ascending=False)
    print("\n[Top null rate %]")
    print(null_rate.head(topn_null))

## 2) orders 전처리 (기준 테이블)
- 문자열 날짜를 datetime으로 변환
- `order_id` 중복 여부 확인

In [5]:
order_dt_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in order_dt_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

audit_table(orders, "orders", key_cols=["order_id", "customer_id"])
print("duplicate order_id:", orders["order_id"].duplicated().sum())
print("\norder_status 분포")
print(orders["order_status"].value_counts(dropna=False))


===== orders =====
shape: (99441, 8)
unique(order_id): 99441 / null: 0
unique(customer_id): 99441 / null: 0

[Top null rate %]
order_delivered_customer_date    2.981668
order_delivered_carrier_date     1.793023
order_approved_at                0.160899
order_id                         0.000000
order_purchase_timestamp         0.000000
order_status                     0.000000
customer_id                      0.000000
order_estimated_delivery_date    0.000000
dtype: float64
duplicate order_id: 0

order_status 분포
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## 3) customers 전처리
- 고객 식별자(`customer_id`, `customer_unique_id`) 점검
- 우편번호/지역 결측 확인

In [6]:
audit_table(customers, "customers", key_cols=["customer_id", "customer_unique_id"])
print("duplicate customer_id:", customers["customer_id"].duplicated().sum())


===== customers =====
shape: (99441, 5)
unique(customer_id): 99441 / null: 0
unique(customer_unique_id): 96096 / null: 0

[Top null rate %]
customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64
duplicate customer_id: 0


## 4) order_items 전처리
- 주문-상품-셀러 연결 핵심 테이블
- 금액 컬럼(`price`, `freight_value`) 품질 점검

In [7]:
audit_table(order_items, "order_items", key_cols=["order_id", "product_id", "seller_id"])
print("price < 0:", (order_items["price"] < 0).sum())
print("freight_value < 0:", (order_items["freight_value"] < 0).sum())


===== order_items =====
shape: (112650, 7)
unique(order_id): 98666 / null: 0
unique(product_id): 32951 / null: 0
unique(seller_id): 3095 / null: 0

[Top null rate %]
order_id               0.0
order_item_id          0.0
product_id             0.0
seller_id              0.0
shipping_limit_date    0.0
price                  0.0
freight_value          0.0
dtype: float64
price < 0: 0
freight_value < 0: 0


## 5) order_payments 전처리
- 주문당 다건 가능
- 결측/이상치 점검 후 `order_id` 기준 집계 테이블 생성

In [8]:
audit_table(order_payments, "order_payments", key_cols=["order_id"])
print("payment_value < 0:", (order_payments["payment_value"] < 0).sum())
print("payment_installments < 0:", (order_payments["payment_installments"] < 0).sum())

pay_agg = order_payments.groupby("order_id", as_index=False).agg(
    payment_value_total=("payment_value", "sum"),
    payment_installments_max=("payment_installments", "max"),
    payment_type_nunique=("payment_type", "nunique"),
    payment_sequential_max=("payment_sequential", "max"),
)
audit_table(pay_agg, "pay_agg", key_cols=["order_id"])


===== order_payments =====
shape: (103886, 5)
unique(order_id): 99440 / null: 0

[Top null rate %]
order_id                0.0
payment_sequential      0.0
payment_type            0.0
payment_installments    0.0
payment_value           0.0
dtype: float64
payment_value < 0: 0
payment_installments < 0: 0

===== pay_agg =====
shape: (99440, 5)
unique(order_id): 99440 / null: 0

[Top null rate %]
order_id                    0.0
payment_value_total         0.0
payment_installments_max    0.0
payment_type_nunique        0.0
payment_sequential_max      0.0
dtype: float64


## 6) order_reviews 전처리
- 날짜형 변환
- 리뷰 점수 범위 점검 후 `order_id` 기준 집계

In [9]:
review_dt_cols = ["review_creation_date", "review_answer_timestamp"]
for c in review_dt_cols:
    order_reviews[c] = pd.to_datetime(order_reviews[c], errors="coerce")

audit_table(order_reviews, "order_reviews", key_cols=["review_id", "order_id"])
print("review_score out of 1~5:",
      ((order_reviews["review_score"] < 1) | (order_reviews["review_score"] > 5)).sum())

rev_agg = order_reviews.groupby("order_id", as_index=False).agg(
    review_score_mean=("review_score", "mean"),
    review_count=("review_id", "count"),
    review_creation_min=("review_creation_date", "min"),
)
audit_table(rev_agg, "rev_agg", key_cols=["order_id"])


===== order_reviews =====
shape: (99224, 7)
unique(review_id): 98410 / null: 0
unique(order_id): 98673 / null: 0

[Top null rate %]
review_comment_title       88.341530
review_comment_message     58.702532
review_id                   0.000000
review_score                0.000000
order_id                    0.000000
review_creation_date        0.000000
review_answer_timestamp     0.000000
dtype: float64
review_score out of 1~5: 0

===== rev_agg =====
shape: (98673, 4)
unique(order_id): 98673 / null: 0

[Top null rate %]
order_id               0.0
review_score_mean      0.0
review_count           0.0
review_creation_min    0.0
dtype: float64


## 7) products + category translation 전처리
- 카테고리 영문명 매핑
- 상품 속성 결측치 확인

In [10]:
products = products.merge(cat_tr, on="product_category_name", how="left")
audit_table(products, "products(+translation)", key_cols=["product_id"])


===== products(+translation) =====
shape: (32951, 10)
unique(product_id): 32951 / null: 0

[Top null rate %]
product_category_name_english    1.890686
product_category_name            1.851234
product_photos_qty               1.851234
product_name_lenght              1.851234
product_description_lenght       1.851234
product_weight_g                 0.006070
product_height_cm                0.006070
product_length_cm                0.006070
product_width_cm                 0.006070
product_id                       0.000000
dtype: float64


## 8) sellers 전처리
- 셀러 키/지역 결측 점검

In [11]:
audit_table(sellers, "sellers", key_cols=["seller_id"])
print("duplicate seller_id:", sellers["seller_id"].duplicated().sum())


===== sellers =====
shape: (3095, 4)
unique(seller_id): 3095 / null: 0

[Top null rate %]
seller_id                 0.0
seller_zip_code_prefix    0.0
seller_city               0.0
seller_state              0.0
dtype: float64
duplicate seller_id: 0


## 9) geolocation 전처리 (집계본 생성)
- 원본은 zip prefix 중복이 많아 바로 조인하면 데이터 폭증 위험
- zip prefix 기준 대표값으로 축약해서 사용

In [12]:
audit_table(geolocation, "geolocation", key_cols=["geolocation_zip_code_prefix"])

geo_zip = geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    geolocation_lat=("geolocation_lat", "mean"),
    geolocation_lng=("geolocation_lng", "mean"),
    geolocation_city=("geolocation_city", "first"),
    geolocation_state=("geolocation_state", "first"),
)
audit_table(geo_zip, "geo_zip(aggregated)", key_cols=["geolocation_zip_code_prefix"])


===== geolocation =====
shape: (1000163, 5)
unique(geolocation_zip_code_prefix): 19015 / null: 0

[Top null rate %]
geolocation_zip_code_prefix    0.0
geolocation_lat                0.0
geolocation_lng                0.0
geolocation_city               0.0
geolocation_state              0.0
dtype: float64

===== geo_zip(aggregated) =====
shape: (19015, 5)
unique(geolocation_zip_code_prefix): 19015 / null: 0

[Top null rate %]
geolocation_zip_code_prefix    0.0
geolocation_lat                0.0
geolocation_lng                0.0
geolocation_city               0.0
geolocation_state              0.0
dtype: float64


## 10) (선택) 조인 직전 준비 완료 체크
- 여기까지 끝나면 테이블 개별 전처리는 완료
- 다음 단계에서 `orders` 중심으로 안전 조인 진행

In [13]:
print("전처리 준비 완료")
print("orders:", orders.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)
print("pay_agg:", pay_agg.shape)
print("rev_agg:", rev_agg.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("geo_zip:", geo_zip.shape)

전처리 준비 완료
orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
pay_agg: (99440, 5)
rev_agg: (98673, 4)
products: (32951, 10)
sellers: (3095, 4)
geo_zip: (19015, 5)


---
# 추가 검증(전처리 QA)
아래 셀들은 기존 전처리 결과를 검증하기 위한 단계입니다.
- 키 무결성
- 조인 전후 행수 검증
- 날짜 논리 검증
- 금액 일관성 검증
- 리뷰/결제 다건 정책 점검
- 분석 제외 조건 점검
---

# =========================================================
# [QA-01] 키 무결성 검증
# =========================================================

In [14]:
# 1) orders.customer_id -> customers.customer_id
missing_customer = ~orders["customer_id"].isin(customers["customer_id"])
print("orders -> customers 미매칭 건수:", missing_customer.sum())

# 2) order_items.order_id -> orders.order_id
missing_order = ~order_items["order_id"].isin(orders["order_id"])
print("order_items -> orders 미매칭 건수:", missing_order.sum())

orders -> customers 미매칭 건수: 0
order_items -> orders 미매칭 건수: 0


## QA-02 조인 전후 행수 검증
- 기준 grain: `order_items` 행 기준
- 조인으로 행이 비정상 증가하지 않는지 확인

In [15]:
# =========================================================
# [QA-02] 조인 전후 행수 검증
# =========================================================
base_rows = len(order_items)

tmp = (order_items
       .merge(orders[["order_id","customer_id"]], on="order_id", how="left")
       .merge(customers[["customer_id"]], on="customer_id", how="left"))

print("order_items 행수:", base_rows)
print("조인 후 행수   :", len(tmp))
print("행수 변화      :", len(tmp) - base_rows)

order_items 행수: 112650
조인 후 행수   : 112650
행수 변화      : 0


## QA-03 날짜 논리 검증
- 시간 순서가 뒤집힌 케이스 탐지

In [16]:
# =========================================================
# [QA-03] 날짜 논리 검증
# =========================================================
c1 = (orders["order_approved_at"] < orders["order_purchase_timestamp"]).sum()
c2 = (orders["order_delivered_carrier_date"] < orders["order_approved_at"]).sum()
c3 = (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum()
c4 = (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum()

print("approved < purchase:", c1)
print("carrier < approved :", c2)
print("customer < carrier :", c3)
print("customer < purchase:", c4)

approved < purchase: 0
carrier < approved : 1359
customer < carrier : 23
customer < purchase: 0


## QA-03 후속 처리(권장)
날짜 역전 케이스를 즉시 삭제하지 않고 `anomaly_flag`로 관리합니다.

- 목적:
  1. 퍼널/코호트 분석은 최대한 데이터 보존
  2. 배송 리드타임 분석에서는 이상치 제외/포함 비교 가능하게 구성
- 기준:
  - `carrier < approved`
  - `customer < carrier`

In [17]:
# =========================================================
# [POST-QA] 날짜 역전 이상치 플래그 생성
# =========================================================
orders_qc = orders.copy()

orders_qc["anomaly_carrier_before_approved"] = (
    orders_qc["order_delivered_carrier_date"] < orders_qc["order_approved_at"]
)

orders_qc["anomaly_customer_before_carrier"] = (
    orders_qc["order_delivered_customer_date"] < orders_qc["order_delivered_carrier_date"]
)

orders_qc["anomaly_flag"] = (
    orders_qc["anomaly_carrier_before_approved"] |
    orders_qc["anomaly_customer_before_carrier"]
).astype("int8")

print("anomaly_flag 분포")
print(orders_qc["anomaly_flag"].value_counts(dropna=False))

print("\n세부 건수")
print("carrier < approved :", orders_qc["anomaly_carrier_before_approved"].sum())
print("customer < carrier :", orders_qc["anomaly_customer_before_carrier"].sum())

anomaly_flag 분포
anomaly_flag
0    98059
1     1382
Name: count, dtype: int64

세부 건수
carrier < approved : 1359
customer < carrier : 23


## 배송 리드타임 파생변수 생성
- 전체 버전(`delivery_days_all`)
- 이상치 제외 버전(`delivery_days_clean`)
- 지연 여부(`is_late`)

In [18]:
# =========================================================
# [POST-QA] 배송 파생변수 생성
# =========================================================
orders_qc["delivery_days_all"] = (
    orders_qc["order_delivered_customer_date"] - orders_qc["order_purchase_timestamp"]
).dt.days

orders_qc["is_late"] = (
    orders_qc["order_delivered_customer_date"] > orders_qc["order_estimated_delivery_date"]
).astype("Int64")

# 이상치 제외 버전
orders_qc["delivery_days_clean"] = orders_qc["delivery_days_all"].where(
    orders_qc["anomaly_flag"] == 0, pd.NA
)

print(orders_qc[["delivery_days_all", "delivery_days_clean"]].describe())
print("\nis_late 분포")
print(orders_qc["is_late"].value_counts(dropna=False))

       delivery_days_all  delivery_days_clean
count       96476.000000         95103.000000
mean           12.094086            12.153844
std             9.551746             9.577745
min             0.000000             0.000000
25%             6.000000             6.000000
50%            10.000000            10.000000
75%            15.000000            15.000000
max           209.000000           209.000000

is_late 분포
is_late
0    91614
1     7827
Name: count, dtype: Int64


## 포함 vs 제외 비교 체크
배송시간 통계가 이상치에 얼마나 민감한지 빠르게 비교합니다.

In [19]:
# =========================================================
# [POST-QA] 포함/제외 비교
# =========================================================
summary = pd.DataFrame({
    "metric": ["count", "mean", "median", "p95", "max"],
    "delivery_all": [
        orders_qc["delivery_days_all"].count(),
        orders_qc["delivery_days_all"].mean(),
        orders_qc["delivery_days_all"].median(),
        orders_qc["delivery_days_all"].quantile(0.95),
        orders_qc["delivery_days_all"].max(),
    ],
    "delivery_clean": [
        orders_qc["delivery_days_clean"].count(),
        orders_qc["delivery_days_clean"].mean(),
        orders_qc["delivery_days_clean"].median(),
        orders_qc["delivery_days_clean"].quantile(0.95),
        orders_qc["delivery_days_clean"].max(),
    ],
})
print(summary)

   metric  delivery_all  delivery_clean
0   count  96476.000000    95103.000000
1    mean     12.094086       12.153844
2  median     10.000000       10.000000
3     p95     29.000000       29.000000
4     max    209.000000      209.000000


## QA-04 금액 일관성 검증
- 주문 아이템 합계와 결제 합계 차이 확인

In [20]:
# =========================================================
# [QA-04] 금액 일관성 검증
# =========================================================
item_sum = (order_items.assign(item_total=order_items["price"] + order_items["freight_value"])
            .groupby("order_id", as_index=False)["item_total"].sum())

amt_check = item_sum.merge(pay_agg[["order_id","payment_value_total"]], on="order_id", how="left")
amt_check["diff"] = amt_check["payment_value_total"] - amt_check["item_total"]

print(amt_check["diff"].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]))
print("절대오차 1 초과 건수:", (amt_check["diff"].abs() > 1).sum())

count    9.866500e+04
mean     2.909228e-02
std      1.129221e+00
min     -5.162000e+01
1%      -5.684342e-14
5%      -1.421085e-14
50%      0.000000e+00
95%      1.421085e-14
99%      5.684342e-14
max      1.828100e+02
Name: diff, dtype: float64
절대오차 1 초과 건수: 249


## QA-05 리뷰/결제 다건 점검
- 주문당 리뷰/결제 레코드 수 분포 확인

In [21]:
# =========================================================
# [QA-05] 다건(1:N) 점검
# =========================================================
pay_cnt = order_payments.groupby("order_id").size()
rev_cnt = order_reviews.groupby("order_id").size()

print("[payments] 주문당 레코드 수 분포")
print(pay_cnt.value_counts().sort_index().head(10))

print("\n[reviews] 주문당 레코드 수 분포")
print(rev_cnt.value_counts().sort_index().head(10))

[payments] 주문당 레코드 수 분포
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
Name: count, dtype: int64

[reviews] 주문당 레코드 수 분포
1    98126
2      543
3        4
Name: count, dtype: int64


## QA-06 분석 제외 조건 점검
- 주문 상태별 분포 확인 후 제외 정책 확정

In [22]:
# =========================================================
# [QA-06] 주문 상태 점검
# =========================================================
print(orders["order_status"].value_counts(dropna=False))

# 예시: 배송/리뷰 분석용 대상
analysis_orders = orders[orders["order_status"].isin(["delivered"])]
print("\n배송/리뷰 분석용 delivered 주문 수:", len(analysis_orders))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

배송/리뷰 분석용 delivered 주문 수: 96478


## QA-07 리뷰 집계 테이블 생성
- `order_reviews`는 주문당 다건 가능하므로 `order_id` 기준 집계 후 조인

In [23]:
# =========================================================
# [QA-07] reviews 집계
# =========================================================
rev_agg = order_reviews.groupby("order_id", as_index=False).agg(
    review_score_mean=("review_score", "mean"),
    review_count=("review_id", "count"),
    review_creation_min=("review_creation_date", "min"),
)

print("rev_agg shape:", rev_agg.shape)
print("rev_agg order_id 중복:", rev_agg["order_id"].duplicated().sum())
rev_agg.head()

rev_agg shape: (98673, 4)
rev_agg order_id 중복: 0


,order_id,review_score_mean,review_count,review_creation_min
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1,2017-09-21
1,00018f77f2f0320c557190d7a144bdd3,4.0,1,2017-05-13
2,000229ec398224ef6ca0657da4fc703e,5.0,1,2018-01-23
3,00024acbcdf0a6daa1e931b038114c75,4.0,1,2018-08-15
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1,2017-03-02


## QA-08 geolocation 집계 테이블 생성
- zip prefix 중복 방지를 위해 대표값으로 축약

In [24]:
# =========================================================
# [QA-08] geolocation 집계
# =========================================================
geo_zip = geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    geolocation_lat=("geolocation_lat", "mean"),
    geolocation_lng=("geolocation_lng", "mean"),
    geolocation_city=("geolocation_city", "first"),
    geolocation_state=("geolocation_state", "first"),
)

print("geo_zip shape:", geo_zip.shape)
print("geo_zip prefix 중복:", geo_zip["geolocation_zip_code_prefix"].duplicated().sum())
geo_zip.head()

geo_zip shape: (19015, 5)
geo_zip prefix 중복: 0


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP


## QA-09 master_df 조인
- 그레인: `order_id + order_item_id`
- 조인 순서: `order_items -> orders -> customers -> sellers -> rev_agg -> geo_zip`

In [25]:
# =========================================================
# [QA-09] 마스터 조인
# =========================================================
master_df = (
    order_items
    .merge(orders, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(sellers, on="seller_id", how="left", suffixes=("", "_seller"))
    .merge(rev_agg, on="order_id", how="left")
    .merge(
        geo_zip,
        left_on="customer_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left"
    )
)

print("master_df shape:", master_df.shape)
print("중복키(order_id, order_item_id):", master_df[["order_id", "order_item_id"]].duplicated().sum())
master_df.head(3)

master_df shape: (112650, 29)
중복키(order_id, order_item_id): 0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state,review_score_mean,review_count,review_creation_min,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,27277,volta redonda,SP,5.0,1.0,2017-09-21,28013.0,-21.762775,-41.309633,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,3471,sao paulo,SP,4.0,1.0,2017-05-13,15775.0,-20.220527,-50.903424,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,37564,borda da mata,MG,5.0,1.0,2018-01-23,35661.0,-19.870305,-44.593326,pará de minas,MG


## QA-10 delivered 필터 + 배송 파생컬럼

In [26]:
# =========================================================
# [QA-10] delivered + 파생
# =========================================================
master_df_delivered = master_df[master_df["order_status"] == "delivered"].copy()

master_df_delivered["delivery_days"] = (
    master_df_delivered["order_delivered_customer_date"]
    - master_df_delivered["order_purchase_timestamp"]
).dt.days

master_df_delivered["approved_to_carrier_days"] = (
    master_df_delivered["order_delivered_carrier_date"]
    - master_df_delivered["order_approved_at"]
).dt.days

master_df_delivered["carrier_to_customer_days"] = (
    master_df_delivered["order_delivered_customer_date"]
    - master_df_delivered["order_delivered_carrier_date"]
).dt.days

master_df_delivered["is_late"] = (
    master_df_delivered["order_delivered_customer_date"]
    > master_df_delivered["order_estimated_delivery_date"]
).astype("Int64")

master_df_delivered["route_state"] = (
    master_df_delivered["seller_state"].fillna("UNK")
    + "->" +
    master_df_delivered["customer_state"].fillna("UNK")
)

master_df_delivered["anomaly_flag"] = (
    (master_df_delivered["order_delivered_carrier_date"] < master_df_delivered["order_approved_at"]) |
    (master_df_delivered["order_delivered_customer_date"] < master_df_delivered["order_delivered_carrier_date"])
).astype("int8")

print("master_df_delivered shape:", master_df_delivered.shape)
master_df_delivered[[
    "order_id","order_item_id","customer_unique_id","seller_state","customer_state",
    "delivery_days","approved_to_carrier_days","carrier_to_customer_days","is_late","anomaly_flag"
]].head(5)

master_df_delivered shape: (110197, 35)


,order_id,order_item_id,customer_unique_id,seller_state,customer_state,delivery_days,approved_to_carrier_days,carrier_to_customer_days,is_late,anomaly_flag
0,00010242fe8c5a6d1ba2dd792cb16214,1,871766c5855e863f6eccc05f988b23cb,SP,RJ,7.0,6.0,1.0,0,0
1,00018f77f2f0320c557190d7a144bdd3,1,eb28e67c4c0b83846050ddfb8a35d051,SP,SP,16.0,8.0,8.0,0,0
2,000229ec398224ef6ca0657da4fc703e,1,3818d81c6709e39d06b2738a8d3a2474,MG,MG,7.0,1.0,6.0,0,0
3,00024acbcdf0a6daa1e931b038114c75,1,af861d436cfc08b2c2ddefd0ba074622,SP,SP,6.0,2.0,4.0,0,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,64b576fb70d441e8f1b2d7d446e483c5,PR,SP,25.0,11.0,13.0,0,0


In [27]:
# =========================================================
# [QA-11] 최종 점검
# =========================================================
print("중복키(order_id, order_item_id):", master_df_delivered[["order_id", "order_item_id"]].duplicated().sum())

check_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
    "customer_unique_id", "customer_state", "seller_state",
    "delivery_days", "is_late", "route_state"
]
print("\n결측률(%)")
print((master_df_delivered[check_cols].isna().mean() * 100).sort_values(ascending=False))

중복키(order_id, order_item_id): 0

결측률(%)
order_approved_at                0.013612
delivery_days                    0.007260
order_delivered_customer_date    0.007260
order_delivered_carrier_date     0.001815
order_purchase_timestamp         0.000000
customer_unique_id               0.000000
order_estimated_delivery_date    0.000000
customer_state                   0.000000
seller_state                     0.000000
is_late                          0.000000
route_state                      0.000000
dtype: float64


## 결측치 처리 방안 (현재 결과 기준)

현재 결측률이 모두 **0.02% 미만**이므로, 강한 결측치 대체/삭제는 필요하지 않습니다.  
임의 보간으로 인한 왜곡 가능성이 더 크기 때문에, **분석 목적별 조건부 제외** 전략을 사용합니다.

### 1) 결측 0% 컬럼
- `order_purchase_timestamp`
- `order_estimated_delivery_date`
- `customer_unique_id`
- `customer_state`
- `seller_state`
- `is_late`
- `route_state`

처리:
- 추가 처리 없음

근거:
- 완전값 컬럼이므로 결측 처리로 인한 이득이 없음

---

### 2) `order_approved_at` (결측률 약 0.0136%)
처리:
- 대체/보간하지 않고 `NaT` 유지

근거:
- 승인 시각은 실제 이벤트 타임스탬프이므로 임의 대체 시
  `approved_to_carrier` 같은 구간 시간 왜곡 가능

---

### 3) `order_delivered_carrier_date` (약 0.0018%),
`order_delivered_customer_date` / `delivery_days` (약 0.0073%)
처리:
- 배송 리드타임 분석에서는 해당 행만 제외
- 전체 주문 분포/건수 분석에서는 유지

근거:
- 결측 비율이 매우 낮아 전체 삭제는 불필요
- 분석 목적별 최소 제외가 정보 손실과 왜곡을 동시에 줄임

---

### 결론
- 본 데이터는 결측치 비율이 매우 낮아 **무리한 대체보다 보수적 처리**가 적절함
- 실무 적용 시 `final_df`(전체 유지)와 `leadtime_df`(리드타임 분석용 결측 제외)를 분리 운영

In [28]:
# =========================================================
# [QA-12] 결측치 처리 정책 적용
# - final_df: delivered 전체 유지
# - leadtime_df: 리드타임 분석에 필요한 날짜 결측만 제외
# =========================================================
final_df = master_df_delivered.copy()

leadtime_df = final_df.dropna(subset=[
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]).copy()

excluded_n = len(final_df) - len(leadtime_df)
excluded_pct = (excluded_n / len(final_df) * 100) if len(final_df) > 0 else 0

print("final_df 행수:", len(final_df))
print("leadtime_df 행수:", len(leadtime_df))
print("리드타임 분석 제외 행수:", excluded_n)
print("리드타임 분석 제외 비율(%):", round(excluded_pct, 6))

final_df 행수: 110197
leadtime_df 행수: 110188
리드타임 분석 제외 행수: 9
리드타임 분석 제외 비율(%): 0.008167


In [29]:
# =========================================================
# [QA-13] 결측률 재확인
# =========================================================
check_cols = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "delivery_days",
]

print("[final_df 결측률(%)]")
print((final_df[check_cols].isna().mean() * 100).sort_values(ascending=False))

print("\n[leadtime_df 결측률(%)]")
print((leadtime_df[check_cols].isna().mean() * 100).sort_values(ascending=False))

[final_df 결측률(%)]
order_approved_at                0.013612
order_delivered_customer_date    0.007260
delivery_days                    0.007260
order_delivered_carrier_date     0.001815
dtype: float64

[leadtime_df 결측률(%)]
order_approved_at                0.013613
order_delivered_carrier_date     0.000000
order_delivered_customer_date    0.000000
delivery_days                    0.000000
dtype: float64


In [30]:
# =========================================================
# [QA-14] 분석용 데이터셋 분리 저장(선택)
# =========================================================
final_df.to_csv("../../../data/final_df_delivered_all.csv", index=False)
leadtime_df.to_csv("../../../data/final_df_delivered_leadtime.csv", index=False)

print("저장은 필요 시 주석 해제해서 실행하세요.")

저장은 필요 시 주석 해제해서 실행하세요.
